# 01 — Exploratory Data Analysis (EDA)
**Group 6 | Topic 6: Market Basket Analysis on Online Retail II | `SEED = 5018`**

This notebook explores the raw **UCI Online Retail II** dataset (~1.06M transaction records) before cleaning and invoice-level sampling.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add repository root to sys.path so we can import from src/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import GROUP_SEED
from src.download_data import download_dataset
from src.preprocessing import load_raw_transactions

# Fix random seed for reproducibility
SEED = GROUP_SEED  # 5018
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT} | SEED = {SEED}")

## 1. Load Raw Dataset & Inspect Schema

In [ ]:
# Download or locate dataset inside data/ and load both annual Excel sheets
dataset_path = download_dataset(data_dir=PROJECT_ROOT / "data")
raw_df = load_raw_transactions(dataset_path)

print(f"Total raw rows: {len(raw_df):,} | Columns: {list(raw_df.columns)}")
raw_df.head()

## 2. Data Quality, Missing Values & Cancelled Orders (`'C'` Prefix)

In [ ]:
# Summarize missing values per column
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_pct": (raw_df.isna().mean() * 100).round(2),
    "dtype": raw_df.dtypes.astype(str),
})
display(missing_summary)

# Count cancelled orders (InvoiceNo starting with 'C') and invalid numeric values
invoice_strings = raw_df["InvoiceNo"].astype(str).str.strip().str.upper()
is_cancelled = invoice_strings.str.startswith("C")
quantities = pd.to_numeric(raw_df["Quantity"], errors="coerce")
prices = pd.to_numeric(raw_df["UnitPrice"], errors="coerce")

print(f"Cancelled rows ('C' prefix): {is_cancelled.sum():,} ({is_cancelled.mean() * 100:.2f}%)")
print(f"Non-positive Quantity rows: {(quantities <= 0).sum():,}")
print(f"Non-positive UnitPrice rows: {(prices <= 0).sum():,}")

## 3. Basket Size & Product Frequency Distributions

In [ ]:
# Preview valid non-cancelled rows for initial basket size inspection
valid_preview = raw_df.loc[~is_cancelled].dropna(subset=["InvoiceNo", "Description"])
basket_sizes = valid_preview.groupby("InvoiceNo")["Description"].nunique()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(basket_sizes.clip(upper=100), bins=40, ax=axes[0], color="steelblue")
axes[0].set_title("Unique Products per Invoice (Clipped at 100)")
axes[0].set_xlabel("Number of Distinct Items in Basket")

top_items = valid_preview["Description"].str.strip().str.upper().value_counts().head(15)
sns.barplot(x=top_items.values, y=top_items.index, ax=axes[1], color="teal")
axes[1].set_title("Top 15 Most Frequent Products (Non-Cancelled)")
axes[1].set_xlabel("Line-Item Count")

plt.tight_layout()
plt.show()